# 302 — MCP Client and Agents

Demonstrates Phase 3 agent operation through the MCP tool layer.

All three specialist agents now receive an `MCPToolClient` instead of direct tool class
instances — no agent holds a reference to `GraphTools`, `RiskTools`, or `TraceTools`.

| Agent | MCP tools called |
|---|---|
| `GraphAgent` | `entity_lookup`, `company_profile`, `expand_ownership`, `shared_address_check`, `sic_context` |
| `RiskAgent` | `ownership_complexity_check`, `control_signal_check`, `address_risk_check`, `industry_context_check` |
| `TraceAgent` | `retrieve_trace`, `find_traces_by_entity` |

In [1]:
import sys
sys.path.insert(0, "..")

import uuid

from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.clients.mcp_tool_client import MCPToolClient
from src.agents.graph_agent import GraphAgent
from src.agents.risk_agent import RiskAgent
from src.agents.trace_agent import TraceAgent
from src.domain.models import InvestigationTrace

# Neo4j connection — used only for TraceService (writing trace events)
settings = get_neo4j_settings()
neo4j = Neo4jRepository(**vars(settings))
neo4j.test_connection()
repo = TraceRepository(neo4j)
trace_service = TraceService(repo)

# Optional AI client
try:
    from src.clients.anthropic_client import AnthropicClient
    ai_client = AnthropicClient()
    print("AI client: ready")
except Exception:
    ai_client = None
    print("AI client: not configured — deterministic summaries only")

# MCP client — manages its own Neo4j connection lazily via the server layer
mcp = MCPToolClient()
print(f"MCP client: {len(mcp.list_tools())} tools registered")

# Agents — no tool class references, only mcp + trace_service
graph_agent = GraphAgent(mcp, trace_service, ai_client)
risk_agent  = RiskAgent(mcp, trace_service, ai_client)
trace_agent = TraceAgent(mcp, trace_service, ai_client)

COMPANY = "VODAFONE 2."
print(f"\nTarget company: {COMPANY!r}")

AI client: ready
MCP client: 15 tools registered

Target company: 'VODAFONE 2.'


## GraphAgent via MCP

Calls `expand_ownership` through `MCPToolClient`. All tool access goes through the MCP layer — the agent has no direct reference to `GraphTools`.

In [2]:
trace_graph = InvestigationTrace(
    request_id=str(uuid.uuid4()),
    entity_name=COMPANY,
    user_id="analyst-302",
)
repo.save_trace(trace_graph)

graph_result = graph_agent.run(
    "expand_ownership",
    {"company_name": COMPANY, "max_depth": 3},
    trace_graph,
)

print(f"GraphAgent  success={graph_result.success}")
print(f"Summary     : {graph_result.summary}")
print(f"tools_used  : {graph_result.tools_used}")
print(f"Trace events: {len(trace_graph.events)}")
if graph_result.success:
    data = graph_result.findings.get("expand_ownership", {})
    paths = data.get("paths", []) if data else []
    ubos  = data.get("ultimate_owners", []) if data else []
    print(f"Ownership hops: {len(paths)}, UBOs: {len(ubos)}")

[03/23/26 19:25:31] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=403771;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=796077;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

GraphAgent  success=True
Summary     : The company VODAFONE 2. has multiple layers of corporate ownership across three levels, but the analysis failed to identify any individual ultimate beneficial owners, which represents a significant gap in beneficial ownership transparency.
tools_used  : ['expand_ownership']
Trace events: 2
Ownership hops: 6, UBOs: 0


## RiskAgent via MCP

Calls `ownership_complexity_check` through `MCPToolClient`.

In [3]:
trace_risk = InvestigationTrace(
    request_id=str(uuid.uuid4()),
    entity_name=COMPANY,
    user_id="analyst-302",
)
repo.save_trace(trace_risk)

risk_result = risk_agent.run(
    "ownership_complexity_check",
    {"company_name": COMPANY},
    trace_risk,
)

print(f"RiskAgent   success={risk_result.success}")
print(f"Summary     : {risk_result.summary}")
print(f"tools_used  : {risk_result.tools_used}")
print(f"Trace events: {len(trace_risk.events)}")
if risk_result.success:
    data = risk_result.findings.get("ownership_complexity_check", {})
    print(f"Risk level  : {data.get('risk_level') if data else 'n/a'}")

RiskAgent   success=True
Summary     : Ownership chain for 'VODAFONE 2.': max depth 3, 3 unique owner(s), 0 individual UBO(s). Complexity risk: HIGH. All chains terminate at corporate entities — no individual UBOs resolved.
tools_used  : ['ownership_complexity_check']
Trace events: 1
Risk level  : HIGH


## TraceAgent via MCP

Uses `find_traces_by_entity` to find the traces created by the cells above, then retrieves one with `retrieve_trace`.

The TraceAgent has its own *operational* trace (`trace_ta`) distinct from the traces it retrieves.

In [4]:
trace_ta = InvestigationTrace(
    request_id=str(uuid.uuid4()),
    entity_name=COMPANY,
    user_id="analyst-302",
)
repo.save_trace(trace_ta)

find_result = trace_agent.run(
    "find_traces_by_entity",
    {"entity_name": COMPANY},
    trace_ta,
)

print(f"TraceAgent  success={find_result.success}")
print(f"Summary     : {find_result.summary}")
print(f"tools_used  : {find_result.tools_used}")
print(f"Trace events: {len(trace_ta.events)}")

found_traces = find_result.findings.get("find_traces_by_entity", []) or []
print(f"Traces found: {len(found_traces)}")
for t in found_traces:
    print(f"  {t['trace_id']}  query={t['query']}")

TraceAgent  success=True
Summary     : Found 3 trace(s) connected to 'VODAFONE 2.'.
tools_used  : ['find_traces_by_entity']
Trace events: 1
Traces found: 3
  4e27525e-7d84-4f85-b944-852a6b45205a  query=VODAFONE 2.
  84142ba1-e558-4ce4-a239-5938baf24db8  query=VODAFONE 2.
  a0b813c8-62bb-41ef-9531-9f23a417af85  query=VODAFONE 2.


In [5]:
# Retrieve the graph-agent trace (the first found trace that isn't our own)
target_id = next(
    (t["trace_id"] for t in found_traces if t["trace_id"] != trace_ta.request_id),
    None,
)

if target_id:
    retrieve_result = trace_agent.run(
        "retrieve_trace",
        {"trace_id": target_id},
        trace_ta,
    )
    print(f"retrieve_trace  success={retrieve_result.success}")
    print(f"Summary         : {retrieve_result.summary}")
    print(f"tools_used      : {retrieve_result.tools_used}")
    trace_data = retrieve_result.findings.get("retrieve_trace")
    if trace_data:
        print(f"Events in target trace: {len(trace_data.get('events', []))}")
        for e in trace_data.get("events", []):
            print(f"  [{e.get('event_type')}] {e.get('tool_name','')} — {e.get('output_summary','')[:80]}")
else:
    print("No prior traces found — run a GraphAgent or RiskAgent first.")

retrieve_trace  success=True
Summary         : Trace '4e27525e-7d84-4f85-b944-852a6b45205a' for 'VODAFONE 2.': 2 event(s).
tools_used      : ['retrieve_trace']
Events in target trace: 2
  [tool_returned] expand_ownership — Found 6 ownership hop(s) across depths [1, 2, 3] for 'VODAFONE 2.'. All chains t
  [agent_reasoning]  — 


## Confirm trace events are written

Each agent call should have persisted at least one `TOOL_RETURNED` event to Neo4j.

In [6]:
import pandas as pd

rows = repo.list_traces(limit=10)
print(f"Traces in Neo4j: {len(rows)}")
if rows:
    df = pd.DataFrame(rows)[["trace_id", "query", "started_at", "event_count"]]
    display(df)

Traces in Neo4j: 6


,trace_id,query,started_at,event_count
0,a0b813c8-62bb-41ef-9531-9f23a417af85,VODAFONE 2.,2026-03-23T08:25:31.297294+00:00,2
1,84142ba1-e558-4ce4-a239-5938baf24db8,VODAFONE 2.,2026-03-23T08:25:31.178447+00:00,1
2,4e27525e-7d84-4f85-b944-852a6b45205a,VODAFONE 2.,2026-03-23T08:25:29.659431+00:00,2
3,48b1e085-c991-4d3e-a531-bd4247f21a3b,ZZZZZ_NONEXISTENT_COMPANY_XYZ,2026-03-23T08:11:39.782189+00:00,2
4,6355a523-81f2-41b9-a267-c9cd3137ea7b,Retrieve and summarise the investigation trace...,2026-03-23T08:11:34.230748+00:00,1
5,cb5b4cea-c7d9-4068-bf4d-6705a3c094f4,GLOBAL METALS UK LTD,2026-03-23T08:10:01.059095+00:00,10


## Cleanup

In [7]:
trace_ids = [
    trace_graph.request_id,
    trace_risk.request_id,
    trace_ta.request_id,
]
for tid in trace_ids:
    result = repo.delete_trace(tid)
    print(f"Deleted trace {tid[:8]}…  found={result['found']}  events={result['events_deleted']}")

Deleted trace 4e27525e…  found=True  events=2
Deleted trace 84142ba1…  found=True  events=1
Deleted trace a0b813c8…  found=True  events=2


In [8]:
mcp.close()    # closes the MCP server's internal Neo4j connection
neo4j.close()  # closes the trace_service connection
print("Drivers closed")

Drivers closed
